ollama server


In [ ]:
# Add cloudflare deb
!mkdir -p --mode=0755 /usr/share/keyrings
!curl -fsSL https://pkg.cloudflare.com/cloudflare-public-v2.gpg | tee /usr/share/keyrings/cloudflare-public-v2.gpg >/dev/null
!echo 'deb [signed-by=/usr/share/keyrings/cloudflare-public-v2.gpg] https://pkg.cloudflare.com/cloudflared any main' | tee /etc/apt/sources.list.d/cloudflared.list

# Update system
!sudo apt update
!sudo apt upgrade

# Install requirements
!sudo apt install -y pciutils zstd cloudflared

!apt autoremove

!killall cloudflared

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os, subprocess, threading, time, socket
from google.colab import userdata

os.environ["OLLAMA_HOST"] = "0.0.0.0"
token = userdata.get("CLOUDFLARE_TOKEN")

def wait_port(port, host="127.0.0.1", timeout=60):
    start = time.time()
    while time.time() - start < timeout:
        s = socket.socket()
        s.settimeout(1)
        ok = s.connect_ex((host, port)) == 0
        s.close()
        if ok:
            return True
        time.sleep(0.5)
    return False

def pipe(name, proc):
    for line in iter(proc.stdout.readline, ''):
        if line:
            print(f"[{name}] {line}", end='')

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=os.environ.copy()
)

threading.Thread(target=pipe, args=("OLLAMA", ollama_proc), daemon=True).start()

if not wait_port(11434):
    raise RuntimeError("Ollama not started, port 11434 problem.")

cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "run", "--token", token],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

threading.Thread(target=pipe, args=("CF", cf_proc), daemon=True).start()

print("Ready. Ollama is running, Cloudflare tunnels connected.")

In [ ]:
!ollama help

In [ ]:
!curl -s http://127.0.0.1:11434/api/tags

In [ ]:
!ollama pull llama3.2

In [ ]:
!ollama pull qwen3.5:9b

In [ ]:
!ollama pull phi4-reasoning:14b

In [ ]:
!ollama list